In [2]:
# define tools
from langchain.tools import tool # import tool creating function



In [5]:
from langchain.chat_models import init_chat_model # to use any model of your choice

In [9]:
import langchain_openai

In [11]:
import os

In [15]:
from dotenv import load_dotenv

In [22]:
load_dotenv()

True

In [23]:
OPENAI_API_KEY=os.getenv("OPENAIAPI")

In [24]:
model=init_chat_model("gpt-4o-mini",temperature=0,model_provider="openai",api_key=OPENAI_API_KEY)

In [ ]:
# Tools
# define the tools

In [25]:
@tool
def add(a: int,b: int)-> int:
    "Args a and b: add both"
    return a+b

In [26]:
@tool
def multiply(a: int,b: int)-> int:
    "Args a and b: add both"
    return a*b

In [35]:
# add tools to a set
tools=[add,multiply]

NameError: name 'add' is not defined

In [36]:
tools_by_name={tool.name : tool for tool in tools}

NameError: name 'tools' is not defined

In [37]:
model_with_tools=model.bind_tools(tools) # model gpt4o is binded with the tools

NameError: name 'tools' is not defined

In [ ]:
## State

In [ ]:
#In a graph state is used to track the output that flows across all nodes and edges and is updated based on the execution
# It can be used to store the messages and number of llm calls

In [30]:
from typing_extensions import TypedDict, Annotated
from langchain.messages import AnyMessage
import operator

In [34]:
# class is messagesstate with typeddict basemodel 
# messages and llm are two variables 
# messages has a type of Annotated and add is allowed 
class MessagesState(TypedDict): 
    messages: Annotated[list[AnyMessage],operator.add] 
    llm:int 

In [ ]:
# Define nodes
# Nodes are like the central points that does something. It may be function or tool calling or anything significant. 
# Think a graph like a map. Nodes are like cities where there can be factories or any products getting created. 
# Edges are highways that enables the flow and decides where the products(or the node output goes)

In [41]:
# Model node
# used to call the LLM

from langchain.messages import SystemMessage
def llm_call(state:dict):
    return {
        "messages": [
            model_with_tools.invoke(
                [
            SystemMessage(
                content="You are a helpful assistant to help with arithmetic operations"
            )
                ]
                +state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls',0) + 1 
           
    }

# state is the form of the output that will be returned. It has 2 variables: messages and llm_calls
# Messages: will be appended with the previous (I think so but why is this a need for arithmetic operations?)
# llm_calls: current state gets updated and incremented by 1


In [42]:
# Tools node
# used to call the tool
# this will also return a state and update it. 
# call the tools whatever is necessary and let the llm use it for the operation

from langchain.messages import ToolMessage

def tool_node(state:dict):
    """ to perform the tool call "" but where is the tool called? if a tool is called, you need to pass the messages """
    
    result=[]

    for tool_call in state["messages"][-1].tool_calls:
        tool=tools_by_name[tool_call["name"]]
        observation=tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation,tool_call_id=tool_call['id']))
        return {"messages":result}   

In [46]:
# Define end logic
# The llm can call a tool or reach the end based on whether llm calls a tool or not
from typing import Literal
from langgraph.graph import StateGraph, START, END

In [48]:
def should_continue(state: MessagesState)-> Literal["tool_node",END]:
    """ decide if we should continue the loop based on whether llm called a tool or not """

    messages=state["messages"]
    last_message=messages[-1]

    # If llm makes a tool call, then perform an action: call tool_node
    if last_message.tool_calls:
        return "tool_node"

    # otherwise stop, reply to the user

    return END
